In [1]:
import os
import pandas as pd

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [2]:
import dspy
lm = dspy.LM('openai/gpt-4.1-mini', temperature=1.0, max_tokens=16000, api_key=OPENAI_API_KEY)
dspy.configure(lm=lm)

# Dataset

In [3]:
import dspy
import pandas as pd

def init_dataset_from_df(df, split_ratio=(0.8, 0.1, 0.1), seed=42):
    assert abs(sum(split_ratio) - 1.0) < 1e-6, "split_ratio must sum to 1.0"
    assert {'text', 'polarization'}.issubset(df.columns), "DataFrame must have 'text' and 'polarization' columns"

    # Shuffle deterministically
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    total = len(df)

    # Split boundaries
    train_end = int(split_ratio[0] * total)
    val_end = train_end + int(split_ratio[1] * total)

    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]

    # Convert to DSPy examples
    def to_dspy_list(sub_df):
        return [
            dspy.Example({
                "statement": row["text"],
                "answer": row["polarization"],
            }).with_inputs("statement")
            for _, row in sub_df.iterrows()
        ]

    train_set = to_dspy_list(train_df)
    val_set = to_dspy_list(val_df)
    test_set = to_dspy_list(test_df)

    return train_set, val_set, test_set


# GEPA

In [4]:
from data_utils import split_dataframe

In [5]:
filename = "../data/dev_phase/subtask1/train/eng.csv"
english_train_df = pd.read_csv(filename)
english_train_df = english_train_df.sample(n=400, random_state=42).reset_index(drop=True)

train_df, val_df, test_df = init_dataset_from_df(english_train_df)

In [6]:
len(train_df), len(val_df), len(test_df)

(8, 1, 1)

In [7]:
# Get the first example
example = train_df[0]

# Access its fields
print(example)                # Shows full Example object

Example({'statement': 'IEBC LISTED 1,031,645 new voters in the second phase of the national voter registration exercise that ended on Feb 06.', 'answer': 0}) (input_keys={'statement'})


In [8]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = example["answer"]
    statement = example["statement"]
    llm_answer = prediction.answer

    if llm_answer not in ['0', '1']:
        feedback_text = (
            f"The model must output only '1' or '0' to indicate whether the statement is polarizing. "
            f"You responded with '{prediction.answer}', which is invalid. "
            f"The correct label for this statement is '{correct_answer}'."
        )
        if statement:
            feedback_text += f"\n\nStatement:\n{statement}\n\n"
        feedback_text += (
            "Review the classification logic and ensure the response is strictly 'Yes' or 'No' "
            "without any justification or extra formatting."
        )
        return dspy.Prediction(score=0, feedback=feedback_text)

    # Compute correctness
    score = int(llm_answer == correct_answer)

    # Construct feedback
    if score == 1:
        feedback_text = f"Correct — the statement is labeled '{correct_answer}'."
    else:
        feedback_text = (
            f"Incorrect — the correct label is '{correct_answer}', but you predicted '{llm_answer}'."
        )
        if statement:
            feedback_text += f"\n\nStatement:\n{statement}\n"
        feedback_text += (
            "\nRe-examine the linguistic or emotional cues that indicate polarization. "
            "Polarizing statements often include adversarial framing, group identity references, "
            "or moral/emotional intensifiers. Non-polarizing statements tend to remain descriptive or factual."
        )

    return dspy.Prediction(score=score, feedback=feedback_text)

In [9]:
class GenerateResponse(dspy.Signature):
    statement = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

In [11]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=1,
    track_stats=True,
    reflection_minibatch_size=3,
    reflection_lm=dspy.LM('openai/gpt-4.1-mini', temperature=1.0, max_tokens=16000, api_key=OPENAI_API_KEY)

)

optimized_program = optimizer.compile(
    program,
    trainset=train_df,
    valset=val_df,
)

2025/12/20 22:00:04 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 384 metric calls of the program. This amounts to 42.67 full evals on the train+val set.
2025/12/20 22:00:04 INFO dspy.teleprompt.gepa.gepa: Using 1 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
GEPA Optimization:   0%|          | 0/384 [00:00<?, ?rollouts/s]2025/12/20 22:00:04 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 1 (0.0%)
2025/12/20 22:00:04 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.0
2025/12/20 22:00:04 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.0


Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00, 190.97it/s]

2025/12/20 22:00:04 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)


2025/12/20 22:00:12 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: You will be given an input field named `statement` containing a textual statement. Your task is to analyze the given statement and determine whether it is polarizing or not.

A polarizing statement is one that is likely to cause division, controversy, or strong disagreement among people, often because it expresses a partisan, emotionally charged, or controversial opinion related to political or social issues.

**Output Requirement:**  
You must respond **only** with a single character:  
- Output `1` if the statement is polarizing.  
- Output `0` if the statement is not polarizing.

**Important Constraints:**  
- Do NOT provide any explanation, reasoning, additional text, headers, or formatting.  
- Output exactly one character, either `1` or `0`, with no surrounding whitespace or punctuation.  
- Do not respond with words such as "Yes", "No", "True", "False", or any other synonyms.  
- Igno

Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00, 228.37it/s]

...

2025/12/20 22:13:11 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)


2025/12/20 22:13:16 INFO dspy.teleprompt.gepa.gepa: Iteration 63: Proposed new text for predict: Given an input field named `statement`, your task is to classify whether the statement is polarizing or not. 

Output a single-digit string:
- Output '1' if the statement is polarizing.
- Output '0' if the statement is not polarizing.

A "polarizing" statement is one likely to cause division, controversy, or strong opposing opinions among people. Statements that are neutral, factual, or unlikely to provoke significant disagreement should be labeled '0'.

Important guidelines:

- Your response must be strictly and only '1' or '0'. Do not output explanations, reasoning, summaries, or any additional text or formatting.
- Do not output words such as "Yes", "No", or any kind of justification.
- Consider political, social, or ideologically charged content as potential indicators of polarizing statements.
- Neutral statements presenting straightforward facts without controversy should be labeled '

Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00, 125.95it/s]

2025/12/20 22:13:25 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)


2025/12/20 22:13:31 INFO dspy.teleprompt.gepa.gepa: Iteration 64: Proposed new text for predict: Task Description:

You will be given a single input field called `statement` containing a text string. Your task is to analyze this `statement` and determine whether it is polarizing or not. A polarizing statement is one that is likely to cause division, strong disagreement, or emotional conflict among people, often related to political, social, or cultural topics.

Output Requirements:

- Your output must be a single field named `answer`.
- The value of `answer` must be strictly either:
  - '1' if the statement is polarizing,
  - '0' if the statement is not polarizing.
- You must output only the exact character '1' or '0', without any additional text, explanation, commentary, or formatting.
- Do not output any reasoning, justification, or any other field besides `answer`.

Detailed Instructions and Considerations:

- Evaluate whether the statement is likely to provoke strong opinions, cont